# 🔬 Clase 2: Metodología CRISP-DM, EDA, Preprocesamiento & KNN (Titanic)
**Cátedra:** Minería de Datos y Aprendizaje Automático — Facultad de Informática, UNLP (2026)
**Profesores:** Dr. Franco Ronchetti & Dr. Waldo Hasperué  
**Alumno:** Agustín Barthe  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---
## 🎯 Objetivos de la Práctica
1. Construir la **Vista Minable** a partir de datos crudos heterogéneos del Titanic.
2. Ejecutar un **Análisis Exploratorio de Datos (EDA)** riguroso identificando sesgos muestrales, valores nulos y correlaciones.
3. Implementar la preparación tabular: eliminación de columnas ruidosas, codificación binaria y **One-Hot Encoding**.
4. Aplicar la regla de oro anti **Data Leakage**: ajustar `StandardScaler` exclusivamente con el conjunto de entrenamiento.
5. Evaluar el clasificador **K-Nearest Neighbors (KNN)** y construir un **Pipeline con Validación Cruzada (ShuffleSplit)**.


### 1. Dependencias y Parámetros Globales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

NORMALIZAR = 1
print("✓ Librerías cargadas correctamente.")

---
## 📊 2. Carga del Dataset e Inspección Inicial (EDA)
Cargamos `titanic.csv` e inspeccionamos dimensiones y valores nulos por atributo.

In [ ]:
dataset_path = 'titanic.csv'
df = pd.read_csv(dataset_path)

print(f"Dimensiones iniciales: {df.shape[0]} filas × {df.shape[1]} columnas\n")
print("Valores nulos detectados por columna:")
print(df.isnull().sum())

### Visualizaciones Clave de la Cátedra
Graficamos la distribución de supervivencia según sexo y clase de pasajero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Supervivencia por Sexo
sns.barplot(ax=axes[0], x='Sex', y='Survived', data=df, palette='magma')
axes[0].set_title('Tasa de Supervivencia según Sexo')
axes[0].set_ylabel('Ratio de Supervivencia')

# Supervivencia por Clase (Pclass)
sns.barplot(ax=axes[1], x='Pclass', y='Survived', data=df, palette='crest')
axes[1].set_title('Tasa de Supervivencia según Clase de Pasajero')
axes[1].set_ylabel('Ratio de Supervivencia')

plt.tight_layout()
plt.show()

---
## 🧹 3. Preparación y Limpieza de Datos (Construcción de la Vista Minable)
- Eliminamos columnas irrelevantes o con nulos masivos: `PassengerId`, `Name`, `Ticket`, `Cabin`.
- Removemos filas residuales con nulos mediante `dropna()`.
- Mapeamos la variable binaria `Sex` a `{male: 0, female: 1}`.
- Aplicamos **One-Hot Encoding** a `Embarked` para evitar imponer un orden artificial a los puertos.

In [ ]:
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_clean = df.drop(columns=columns_to_drop).dropna()

# Binarización de Sexo
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})

# One-Hot Encoding en Embarked
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], prefix='Embarked', dtype=float)

# Separación de Matriz de Características X y Target y
X = df_clean.drop('Survived', axis=1)
y = df_clean['Survived']

print(f"Vista minable final: {X.shape[0]} muestras, {X.shape[1]} features")
print("Columnas procesadas:", list(X.columns))

---
## ⚖️ 4. Partición Train / Test & Escalamiento Anti-Leakage

In [ ]:
# Partición estratificada 80/20
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Instancias para Entrenamiento: {x_train.shape[0]}")
print(f"Instancias para Evaluación:   {x_test.shape[0]}")

# Escalamiento riguroso: fit solo en Train
if NORMALIZAR:
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    print("✓ Scaler ajustado exclusivamente sobre Train (cero Data Leakage).")

---
## 🤖 5. Entrenamiento y Evaluación de KNN (K=5)

In [ ]:
K = 5
modelo_knn = KNeighborsClassifier(n_neighbors=K)
modelo_knn.fit(x_train_scaled, y_train)

acc_train = accuracy_score(y_train, modelo_knn.predict(x_train_scaled))
acc_test = accuracy_score(y_test, modelo_knn.predict(x_test_scaled))

print(f"Precisión en TRAIN (K={K}): {acc_train*100:.2f}%")
print(f"Precisión en TEST  (K={K}): {acc_test*100:.2f}%")

---
## 🔄 6. Validación Cruzada con Scikit-Learn Pipeline & ShuffleSplit
Encapsulamos el `StandardScaler` y el `KNeighborsClassifier` en un `Pipeline` para garantizar que en cada uno de los 10 splits de validación aleatoria, el escalador se ajuste únicamente con los datos de entrenamiento de ese fold.

In [ ]:
# Definición del Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

N_SPLITS = 10
ss = ShuffleSplit(n_splits=N_SPLITS, test_size=0.2, random_state=42)

cv_results = cross_validate(pipeline, X, y, cv=ss, scoring=['accuracy'], return_train_score=True)

mean_train = cv_results['train_accuracy'].mean()
std_train = cv_results['train_accuracy'].std()
mean_test = cv_results['test_accuracy'].mean()
std_test = cv_results['test_accuracy'].std()

print(f"Accuracy TRAIN: {mean_train*100:.2f}% (± {std_train*100:.2f}%)")
print(f"Accuracy TEST:  {mean_test*100:.2f}% (± {std_test*100:.2f}%)")

# Gráfico de splits
plt.figure(figsize=(12, 5))
splits = [f'Split {i+1}' for i in range(N_SPLITS)]
plot_df = pd.DataFrame({
    'Split': splits * 2,
    'Accuracy': list(cv_results['train_accuracy']) + list(cv_results['test_accuracy']),
    'Conjunto': ['Train'] * N_SPLITS + ['Test'] * N_SPLITS
})

sns.barplot(data=plot_df, x='Split', y='Accuracy', hue='Conjunto', palette='viridis')
plt.axhline(mean_test, color='coral', linestyle='--', label=f'Test Mean: {mean_test*100:.1f}%')
plt.title('Rendimiento por Split en Validación Cruzada (ShuffleSplit 10x)')
plt.ylabel('Accuracy')
plt.ylim(0.65, 0.95)
plt.legend()
plt.tight_layout()
plt.show()